# ad-creator · dark-grey-metal 스타일 이미지 생성 (Colab)

이 노트북은 **`instagram_dark_grey_metal_v1`** 프리셋으로 카페 광고 이미지를 생성하고,
결과를 눈으로 확인하면서 프롬프트를 반복 개선하기 위한 작업용 노트북입니다.

**4가지 구도, 각자 다른 환경**
- `medium` — 1층 창가 바 좌석. 유리벽 너머 낮 시간대 거리 풍경(`light.jpg` 분석), 실내 인공조명 없이 유리를 통과하는 자연광만. 바 카운터-창밖 비율과 매거진/화분 소품은 `glass_background.jpg` 분석 반영.
- `close_up` — 기존 다크그레이 콘크리트 벽 + 카운터, 제품이 프레임 대부분을 차지하는 타이트한 크롭.
- `aerial` — 완전 수직(90도) 탑다운, 음료·빵 전부 윗면만 보임. 화분+투명케이스 촛불+빵접시가 다이아몬드 배치(`layout.webp` 분석), 배경(블랙 테이블)이 소품보다 훨씬 넓게 나오도록 고정.
- `handheld` — 온도별로 구도 자체가 갈림: **hot**은 블랙 테이블 위에 그대로, **ice**는 손으로 들어 올려 다크그레이 콘크리트 벽 배경. 소품 전혀 없음(음료+손만).

**온도(`BEVERAGE_TEMP`)는 이제 자동 판별** — 업로드한 사진을 Gemini로 1회 분석해서 조용히 hot/ice를 정하고, 사용자에게 확인을 묻지 않습니다. 온도 관련 오류는 사전 확인창이 아니라 **생성 후 Vision QA 단계에서 사후 검출**됩니다.

**컵 디자인은 `CONTAINER_MODE`로 선택**
- `"user_cup"` — 사용자의 원본 컵을 그대로 재구성 (로고 포함)
- `"reference_cup"` — 컵은 이 프리셋 자체 디자인(hot=손잡이 있는 블랙/다크그레이 머그+받침 세트, ice=필수 더블월 유리컵)을 따르고, 음료 내용물(색/얼음/거품)만 사용자 사진에서 가져옴. 두 모드 다 사진은 1장만 필요합니다.

**소품/디테일 규칙** — 여러 차례 실제 생성 결과를 보면서 다듬은 규칙들입니다:
- 메탈 트레이·냅킨 전부 제거. 빵/케익(크루아상류만, 쿠키 금지)은 도자기/사기 접시(플라스틱 금지)에 포크+나이프와 함께
- 포크·나이프는 헤드+손잡이 절반이 접시 위에, 손잡이는 촬영자 쪽을 향하되 뻣뻣한 평행(11자) 배치는 피함
- 아이스 컵은 받침/빨대 금지, 컵 바닥과 테이블 접촉부에 빛이 새는 흰 선(뜨는 느낌) 금지
- 핫 머그+받침은 항상 세트(같은 색), 화이트 금지(블랙/다크그레이만) — 이 색 규칙은 머그·받침에만 한정되고 화분/초/음료 자체 색에는 적용 안 됨
- 화분은 잎이 큰 식물, 채도는 예외적으로 생생하게 유지 (나머지는 무채색에 가까운 톤)
- 로고는 절대 새로 발명하지 않고, 업로드 사진에 실제로 있는 것만 재현

**파이프라인**: 사진 1장 업로드 → 온도 자동 판별(Gemini 1회) → 4구도 프롬프트 조립 → **OpenAI `gpt-image-2`**(medium 품질) 1회씩 생성 → **Gemini(`gemini-3.1-flash-lite`)** Vision QA 1회씩 → 필요 시 `iterate()`로 특정 구도만 재생성.

> API 키 2개(`OPENAI_API_KEY`, `GEMINI_API_KEY`)는 Colab **Secrets(🔑)**에 등록해두고 `userdata.get()`으로 불러옵니다.
> **루프 안전성**: 첫 생성 = OpenAI 4회 + Gemini(QA) 4회 + Gemini(온도판별) 1회 = 9회, 항상 예측 가능. 모델을 여러 개 돌며 자동 재시도하는 코드 없고, `iterate()`도 사람이 셀을 직접 실행할 때만 동작합니다.

> 팀 공유 zip(`ad-creator-0722.zip`)의 `PASSED_PRESETS.json`/`README_AI_HANDOFF_KO.md`를 참고했지만, 이 프리셋은 아직 팀의 0크레딧 검증·정식 사람 검수(`passed_by_user_review`)를 거치지 않은 **작업 중 버전**입니다. `prompting.py`처럼 여러 프리셋이 공유하는 무거운 런타임에 얹지 않고, 같은 결과를 내는 **가볍고 자기완결적인 버전**(`build_prompt()`)으로 따로 구현했습니다 — 핵심 로직만 뽑은 `build_prompt.py`와, 실제로 조립된 프롬프트 예시(`prompt.md`)를 팀 공유용으로 별도 제공합니다.


## 1. 환경 설치

In [ ]:
# Colab 런타임에 필요한 패키지를 설치합니다. (생성: OpenAI / QA: Gemini)
!pip install -q --upgrade openai google-genai pillow


## 2. API 키 설정 (Colab Secrets 사용)

왼쪽 사이드바의 🔑 **Secrets** 탭에서 아래 두 개를 등록하고, 둘 다 이 노트북에 대한 접근(Notebook access) 토글을 켜두세요.
- `OPENAI_API_KEY` — 이미지 생성(`gpt-image-2`)에 사용
- `GEMINI_API_KEY` — Vision QA(`gemini-3.1-flash-lite`)에 사용

등록해두면 아래 셀에서 `userdata.get()`으로 바로 불러옵니다. 코드에 키를 직접 적을 필요가 없습니다.

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI
from google import genai
from google.genai import types as genai_types

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

openai_client = OpenAI()
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("OpenAI client ready (image generation).")
print("Gemini client ready (vision QA).")


## 3. `dark-grey-metal` 프리셋 (4가지 구도 포함)

레퍼런스 사진들을 분석해서 만든 프리셋입니다. `reference-preset.json`과 동일 스키마를 따르고,
**`shot_variants`** 아래에 4가지 구도(미디엄 컷 / 클로즈업 / 항공샷 / 손으로 든 샷)를 정의해뒀습니다.

- **벽/테이블/의자/무드는 4가지 구도 전부 동일**하게 유지되도록, 공용 필드(`scene`, `color`, `lighting`, `tone_contract`)는 그대로 공유하고 **카메라 구도·소품 배치만** `shot_variants`에서 갈립니다.
- `aerial`(항공샷)은 사용자 요청대로 **빵(크루아상류)이 소품으로 포함**되도록 명시했습니다.
- `aerial`/`handheld`는 첨부해주신 샘플 사진(대리석 테이블, 검정 반짝이 벽)의 **구도·소품 배치 논리만** 가져오고, 실제 표면 재질(대리석/검정 반짝임)은 이 프리셋의 그레이 콘크리트/스톤 환경으로 대체하도록 명시했습니다 (요청하신 "벽/테이블/무드 동일 유지" 반영).
- 지난 생성 결과가 실내가 아니라 길거리처럼 나온 문제를 막기 위해, `scene.background`와 `prompt_blocks.negative`에 **"반드시 실내, 길거리/보도블럭/야외 절대 금지"** 문구를 강하게 추가했습니다.

In [ ]:
import json

# 레퍼런스를 분석해 작성한 프리셋 (schemas/reference-preset.schema.json 확장, shot_variants 포함)
PRESET = json.loads(r'''
{
  "$schema": "../../../schemas/reference-preset.schema.json",
  "schema_version": "2.0.0",
  "preset_id": "instagram_dark_grey_metal_v1",
  "display_name": "인스타 에디토리얼 · 다크 그레이 콘크리트 · 실버 메탈 트레이 카페 장면 v1",
  "concept": "dark_grey_concrete_wall_black_plate_no_tray_industrial_cafe_moment",
  "source": {
    "reference_image": "user-supplied: grey_wall_metal_tray.jpg, black_table_chair.png, props_bread_pot.jpg",
    "supporting_group": "dark-grey-metal/concrete/tray/single/v1",
    "width_px": 1200,
    "height_px": 1600
  },
  "runtime_input_policy": {
    "copyright_rule": "Use only the measured wall/table texture family, tray geometry, camera framing, light distribution and restrained cool-grey palette. Regenerate every scene pixel; never reproduce identifiable background objects, cup/vessel identity, card typography, magazine text or exact shadow shapes from the reference photos.",
    "default_generation_mode": "user_image_plus_abstracted_preset",
    "generation_inputs": [
      "user_product_identity"
    ],
    "reference_usage": "offline_feature_extraction_only"
  },
  "style_abstraction": {
    "allowed_style_features": [
      "one small exact user beverage standing directly on the table/counter/windowsill surface -- no metal tray, no saucer or coaster beneath it",
      "dark cool-grey rough-troweled concrete/plaster wall filling the upper 55-65% of frame as negative space",
      "a slightly different, coarser dark-grey stone/concrete counter surface below a soft horizon seam where wall meets table",
      "a small matte black ceramic plate (holding the croissant/cake prop and a simple fork+knife, handles toward the viewer) placed beside the beverage, with nothing underneath the plate",
      "optional single generic pastry or small potted green plant as a secondary prop, echoing the windowsill reference",
      "soft directional daylight from upper-left/upper-right raking across the concrete, producing gentle matte sheen on the counter surface and a single soft contact shadow beneath the beverage",
      "far, heavily out-of-focus hint of black metal cafe furniture (thin bentwood-style chair legs or a dark wingback silhouette) only in wide establishing variants, never sharp"
    ],
    "forbidden_copy_features": [
      "reference cup/tumbler identity, exact ceramic shape or handle-less silhouette",
      "reference latte-art pattern, exact coffee color/foam structure",
      "exact origin card typography, kerning or wording (\"ETHIOPIA...\" wording is off-limits, replace with generic/blank or brand mark)",
      "exact window signage text or mirrored lettering from the windowsill reference",
      "exact magazine page content, spoon placement or croissant crumb pattern",
      "exact wingback armchair tufting pattern, bistro chair identity or diamond-plate floor texture",
      "exact crop, water-droplet position, aggregate speckle pattern or shadow silhouette"
    ],
    "novel_scene_required": true,
    "runtime_reference_image": false
  },
  "sampling_ranges": {
    "camera_pitch_degrees": [
      15,
      35
    ],
    "focal_length_mm": [
      35,
      50
    ],
    "light_softness": [
      0.55,
      0.75
    ],
    "negative_space_ratio": [
      0.55,
      0.68
    ],
    "prop_count": [
      1,
      3
    ],
    "subject_center_x": [
      0.62,
      0.74
    ],
    "subject_center_y": [
      0.66,
      0.78
    ],
    "subject_width_ratio": [
      0.3,
      0.4
    ],
    "subject_height_ratio": [
      0.16,
      0.22
    ],
    "asymmetry_probability": 1.0,
    "asymmetry_sources": [
      "product pushed toward lower-right third, leaving a tall quiet wall plane upper-left"
    ],
    "pov_modes": [
      "slightly-above quiet tabletop observation, 15-35 degrees"
    ],
    "micro_moments": [
      "one drink standing directly on the counter/table against a bare textured wall, a beat before or after being picked up"
    ]
  },
  "tone_contract": {
    "background_lightness": [
      0.05,
      0.14
    ],
    "background_saturation": [
      0.02,
      0.08
    ],
    "background_plane": "matte cool-grey concrete wall above a subtly coarser dark-grey stone counter, separated by a soft horizontal seam",
    "color_bias": "near-desaturated cool grey throughout; the exact user beverage keeps the only strongly saturated color in frame",
    "global_contrast": [
      0.42,
      0.58
    ],
    "shadow_density": [
      0.68,
      0.85
    ],
    "highlight_behavior": "a soft, restrained rim highlight on the cup and a faint sheen on the black ceramic plate -- highlights stay muted and low-key, never bright or glowing, no blown highlights, no glossy wall reflections",
    "forbidden_surface_reading": [
      "warm beige or brown wall cast",
      "polished/glossy poured concrete",
      "colorful or patterned tile",
      "bright white studio backdrop",
      "any metal serving tray of any kind (no tray anywhere in this preset)",
      "visible large-format aggregate or brick joints"
    ]
  },
  "output": {
    "compatible_aspect_ratios": [
      "4:5",
      "3:4"
    ],
    "preferred_aspect_ratio": "4:5"
  },
  "subject_layout": {
    "main_subject": {
      "role": "one_exact_user_beverage_natively_reconstructed_standing_directly_on_the_surface",
      "bbox": {
        "x_ratio": 0.5,
        "y_ratio": 0.58,
        "width_ratio": 0.34,
        "height_ratio": 0.19
      },
      "center": {
        "x_ratio": 0.67,
        "y_ratio": 0.68
      },
      "height_ratio": 0.19,
      "area_ratio": 0.0646
    },
    "negative_space": {
      "top_ratio": 0.55,
      "left_ratio": 0.12,
      "right_ratio": 0.06,
      "bottom_ratio": 0.12
    }
  },
  "composition": {
    "shot_type": "environment-led single beverage standing directly on a table/counter/windowsill against a tall bare concrete wall, no tray",
    "camera_angle": "fifteen-to-thirty-five degrees downward, near eye-level tabletop observation, not overhead flat-lay",
    "camera_height": "approximately ninety-five to one hundred twenty centimeters",
    "focal_length_equivalent_mm": 40,
    "alignment": "product sits lower-right, balanced by a tall quiet wall plane upper-left occupying more than half the frame",
    "depth_of_field": "the cup and nearest wall/counter texture resolve with crisp micro-contrast; the counter surface softens gently toward the bottom edge",
    "horizon_visibility": "a soft horizontal seam where the vertical wall meets the coarser counter plane anchors the space without a hard architectural line"
  },
  "lighting": {
    "type": "single soft natural daylight source, window-diffused, no visible fixture -- from a time of day when the sun is gentle and the room reads dim and moody: early morning, late afternoon, or an overcast/cloudy moment, NEVER a bright high midday sun and never an airy bright exposure",
    "direction": "upper-left raking across the wall and counter at a shallow angle",
    "azimuth_degrees": 300,
    "elevation_degrees": 35,
    "intensity": "extremely low, very deeply dim and moody -- roughly HALF as bright as the already-dim exposure used previously (so about a quarter of a normal soft-light exposure overall), a heavily underexposed, almost silhouette-leaning look across the whole frame, with just enough light to make out shapes and textures -- not a bright airy look under any circumstance",
    "contrast": "medium: the wall texture reads clearly but shadow edges stay soft, never a hard sharp sun-shadow",
    "softness": 0.66,
    "shadow": "one soft, short contact shadow beneath the cup, edges gently diffused",
    "highlight_control": "protect the beverage's own tonal identity with a soft rim highlight; never let the wall clip to white"
  },
  "color": {
    "temperature": "cool-neutral daylight, no warm filter",
    "white_balance_kelvin": 5600,
    "black_point": "deep neutral charcoal with visible concrete grain, never a crushed pure black",
    "contrast": "weighted lower-mid shadow falloff with a soft highlight shoulder on the cup and black ceramic plate",
    "saturation": "near-monochrome room and surfaces; the exact user beverage is the only saturated color note from the product itself. EXCEPTION: any potted plant's leaves in the scene keep a vivid, healthy, natural green saturation -- do NOT desaturate the plant's foliage to match the muted palette; the leaves should look alive and richly green, not faded, dusty, or grey-green.",
    "palette_hex": [
      "#4A4B4D",
      "#6B6C6E",
      "#8C8D8F",
      "#B9BABC",
      "#2E2F31"
    ]
  },
  "scene": {
    "surface": "no metal tray anywhere -- the beverage stands directly on a coarse dark-grey stone/concrete counter (or, per shot-specific overrides, a black window table or black bistro table). Any food prop sits separately, plated on a small black ceramic plate.",
    "background": "an interior cafe corner: a tall, bare, rough-troweled dark-grey concrete or lime-plaster wall with fine aggregate speckle, occasional small pitting, faint vertical water-stain streaks and one faint natural moisture mark for authenticity, meeting a coarser dark-grey stone/concrete counter below. This is unmistakably an indoor cafe environment, never an outdoor street, sidewalk, pavement, curb, parking lot or exterior wall.",
    "background_complexity": 0.3,
    "texture": "directional hand-trowelled concrete grain on the wall, a coarser pitted stone grain on the counter, matte ceramic on the cup and on the black plate, and finite social-photo acuity",
    "max_prop_count": 2,
    "props": [
      "one small matte black ceramic plate holding one croissant-type pastry or cake plus a simple fork and knife (handles toward the viewer, never hidden under the pastry)",
      "optional one generic pastry on a small plate, or one small potted green plant in a plain white pot, used sparingly and never both at once",
      "optional, far out-of-focus hint of black metal cafe furniture only in wide establishing crops, never a sharp secondary subject"
    ]
  },
  "capture": {
    "grain": "faint low-contrast luminance texture with finite twelve-to-sixteen-megapixel acuity",
    "look": "quiet, moody, industrial-editorial cafe photograph; considered but not overly art-directed",
    "realism": "one camera, one exposure and one physical light system connect wall, counter, plate and vessel",
    "imperfections": [
      "subtle concrete aggregate speckle variation",
      "one faint water-mark or moisture variation on the wall",
      "slightly unequal plate-to-cup spacing",
      "gentle far-corner microcontrast loss",
      "CRITICAL: at the exact point where the glass or mug base touches the table/counter surface, there must be NO bright, white, or transparent gap-line separating the vessel from the surface -- light must not leak in under the base. The contact shadow there should be tight, dark, and continuous, so the vessel reads as physically resting on the surface, never as floating or cut out and pasted on top of it."
    ]
  },
  "preservation_policy": {
    "hard_lock": [
      "exact user product count one",
      "user beverage recipe, color, temperature and serving state exactly as shown in the product photo (ice level and no-straw rule if the beverage is iced; foam/latte-art and steam-appropriate look if the beverage is hot -- never invent the opposite temperature's traits)",
      "source cup semantics reconstructed natively, never pasted or traced from any reference",
      "the beverage stands directly on the surface -- no metal tray and no saucer/coaster beneath it, in every shot",
      "target brand contract",
      "product width 0.30-0.40 and height 0.16-0.22 of frame",
      "NEVER generate, invent, or design a new logo, wordmark, or brand mark of any kind, for either a hot or an iced beverage. The ONLY logo allowed anywhere on the cup is the exact logo that is already visibly present on the cup in the user's uploaded product photo -- reproduce it clearly and legibly on the new vessel shape (mug or glass) exactly as it appears in that photo, with no alterations. If the user's uploaded product photo shows NO logo at all, then the generated cup must also show NO logo -- do not add one. The cup's physical vessel shape may still be adapted (mug for hot / glass for iced) to match the new scene's mood, but the logo content itself is never invented, never modified, and never added where none existed."
    ],
    "editable": [
      "every output pixel",
      "physically consistent perspective, refraction, contact and shadow",
      "minor product x/y position on the surface",
      "concrete speckle phase, moisture-mark position and prop choice",
      "the cup/vessel's physical shape and material (mug vs. glass) to match the scene's mood and the beverage's temperature, while keeping any existing logo/wordmark intact and legible"
    ],
    "reference_exclusions": [
      "reference cup/tumbler identity and exact object pixels",
      "reference latte-art pattern and coffee color",
      "reference card typography, wording or kerning",
      "reference window signage text",
      "exact crop, furniture, floor texture or shadow silhouette"
    ]
  },
  "prompt_blocks": {
    "look": "Natural but art-directed Instagram cafe observation in a dark, moody industrial register: matte cool-grey concrete, brushed steel, restrained daylight, finite acuity.",
    "composition": "Keep exactly one exact user beverage small, standing directly on the surface (no tray, no saucer/coaster) in the lower-right two-thirds of frame. Leave a tall, bare, quiet wall or background plane upper-left as negative space, keeping the composition airy and quiet, Instagram-editorial rather than catalog-centered.",
    "input_roles": "Image one supplies exact product identity. Any additional reference imagery supplies only abstracted wall/background texture, camera framing and light distribution -- never literal object pixels, and never a metal tray. The generated environment (wall, counter, tray, and where applicable table/chairs) must stay the SAME indoor cafe corner described in this preset across every shot type; only the camera framing and prop arrangement change between shot types.",
    "preservation": "Reconstruct the user's cup/container design natively for the new camera and light; never paste, trace or preserve any source boundary. Preserve beverage and serving semantics and obey only the target brand contract. CRITICAL, anti-composite rule: the beverage's surface highlights, reflections, and any visible sheen on the glass or ceramic must be re-rendered as if lit by THIS scene's own light source (the dim, moody interior light or the evening window light described above) -- do NOT copy over the bright, even, studio-style highlights or reflections from the user's original product photo. If the original photo's lighting looks brighter or more evenly lit than this scene, the beverage's highlights must be dimmed and reshaped to match this scene's darker, more directional light, so the drink and its environment read as one single continuous photograph under one light source, never as a brighter object pasted onto a darker background.",
    "negative": "No extra beverage, no metal tray of any kind anywhere, no saucer/coaster under an iced beverage, no glossy or brightly lit wall, no warm beige cast, no legible card or window text, no busy or sharp background furniture, no oversized product, no crushed blacks or blown highlights, no invented logos, no cookies or biscuits as the food prop (croissant-type pastry or cake only). CRITICAL: this is an INDOOR cafe corner, never an outdoor scene -- no street pavement, no sidewalk tile, no exterior curb, no parking lot asphalt, no visible sky (except the blurred evening cityscape specifically described for the medium shot), no daylight horizon, no outdoor foliage or building facades up close. The wall must read as the described rough dark-grey interior concrete/plaster (or the medium shot's glass wall), not a smooth painted exterior wall or a different color/texture entirely."
  },
  "quality_gates": {
    "preset_adherence": "35-50mm lens, 15-35 degree pitch, no-tray direct-surface placement, small-product bounds and cool-grey concrete tone agree",
    "product_identity": "exactly one beverage keeps its recipe and serving state while its cup is physically reconstructed",
    "logo_text": "no visible logo or legible text unless an explicit target brand contract separately authorizes it",
    "artifact_policy": "reject extra beverages, any metal tray, any saucer/coaster under an iced beverage, glossy/mirror-finish surfaces, warm-cast wall, product enlargement, pasted edges, wrong serving state, uniform flat lighting, copied reference props, or cookies used as the food prop"
  },
  "failure_recovery": [
    "restore one exact product standing directly on the surface with no metal tray and no saucer/coaster beneath it",
    "restore the tall bare concrete wall negative-space plane occupying more than half the frame",
    "return the exact product to width 0.30-0.40 and height 0.16-0.22",
    "remove any legible card/window text or reference latte-art pattern",
    "restore cool-neutral desaturated grading and remove any warm beige cast",
    "soften or remove any sharp background furniture leaking into focus"
  ],
  "shot_variants": {
    "medium": {
      "display_name": "미디엄 컷 (1층 창가 바 좌석)",
      "description": "A ground-floor window-bar-seat composition: the viewer is seated at a black bar counter along a glass wall, looking out at a soft-focus daytime street-level scene -- referencing the light_sample photo's exterior (a street-level plaza, blurred building facades, natural daylight), not a night skyline. Lighting is natural daylight passing through the glass, with no interior lamp or pendant fixture of any kind -- referencing the glass_background sample photo's bar-counter-to-exterior ratio and its non-beverage props (a bare folded-open magazine (nothing resting on it), a small potted plant).",
      "camera": "35-50mm equivalent, 15-35 degrees downward, camera at seated bar-stool eye height (~110-130cm).",
      "framing": "The product occupies roughly 30-40% of frame width in the lower-right area, standing directly on the black bar counter with nothing beneath it. Following the glass_background sample's ratio, the black bar counter/tabletop fills roughly the lower 35-45% of the frame, and the glass wall with its soft-focus exterior view fills the remaining upper 55-65% as quiet negative space.",
      "environment_override": {
        "wall_or_background": "A full glass wall/window divided into slim black metal mullions, at ground-floor street level (not a high-rise). Beyond the glass, a softly out-of-focus DAYTIME street-level scene is visible -- light stone or concrete paving, indistinct blurred building facades in muted warm-neutral tones, perhaps a hint of an outdoor bistro table or stool -- suggesting a real ground-floor street view in daylight, not sharp or legible. No legible text, signage, lettering or readable words appear anywhere on the glass. No interior pendant lamp, hanging bulb, or wire cage light fixture of any kind is visible anywhere in this frame. CRITICAL: the interior brightness in this shot must MATCH the same natural-light exposure level used in the other three shots (close_up, aerial, handheld) -- not darker, not dimmer than them. The bar counter, plate, and beverage must be clearly and evenly lit, so the space reads as an open, currently-lit cafe -- never as a closed, unlit, or after-hours-looking store.",
        "surface": "a black-stained wood bar-height counter/table running along the base of the glass wall, narrower than a full windowsill -- the kind of counter with bar stools where a single person sits facing the window."
      },
      "props": [
        "one small matte black ceramic plate with one croissant-type pastry or cake and a simple fork+knife (handles toward the viewer), per the shared prop_plating rule",
        "optional one folded-open magazine or notebook, referencing the glass_background sample -- the magazine/notebook page itself must be completely bare, with NOTHING resting on top of it (no spoon, no utensil, no other object)",
        "optional one small potted plant with large, broad leaves (e.g. a rubber-plant/peperomia type, per the pot sample photo), softly out of focus, at the far edge of frame"
      ],
      "negative_extra": "No rough grey concrete wall in this shot -- the background must be the ground-floor glass wall described above. No night skyline or city lights -- this must read as DAYTIME. No legible window signage or text. No harsh direct sunlight anywhere in this shot. CRITICAL: no interior artificial light source of any kind (no lamp, pendant, bulb, or wire cage fixture) -- the only light in this scene is natural daylight passing through the glass. No metal tray. No cookie as the food prop. CRITICAL: do not underexpose this shot relative to the other three -- match their brightness, this must not look like a closed/unlit store. CRITICAL: at the exact point where the glass or mug base touches the bar counter, there must be NO bright, white, or transparent gap-line -- the vessel must read as physically resting on the counter, never floating or pasted on top of it.",
      "uses_shared_layout": false
    },
    "close_up": {
      "display_name": "클로즈업",
      "description": "A tight, intimate crop on the beverage as it stands directly on the counter (no tray). The wall and counter are still the same rough dark-grey concrete family, but now mostly soft-focus at the edges of frame rather than a large negative-space plane -- the product itself is the dominant subject.",
      "camera": "60-90mm equivalent macro-leaning lens, 20-35 degrees downward, camera height ~40-60cm above the counter.",
      "framing": "The beverage fills roughly 55-70% of frame width and 45-60% of frame height, centered slightly right of frame, standing directly on the counter with nothing beneath it. Only slivers of the concrete wall (upper corner) and stone counter (lower edge) remain visible, both softly defocused. Shallow depth of field resolves the glass/cup, ice, and counter surface crisply while the wall texture behind blurs into soft grey bokeh.",
      "props": [
        "at most one small matte black ceramic plate (croissant-type pastry or cake, per the shared prop_plating rule) partially visible at the frame edge, softly out of focus"
      ],
      "uses_shared_layout": false
    },
    "aerial": {
      "display_name": "항공샷 (완전 수직, 음료·빵 윗면만)",
      "description": "A strict, perfectly vertical (90-degree) top-down view directly above a black round table, framed so that ONLY the top surfaces of the beverage and the food prop are visible -- the camera looks straight down the mouth of the glass/cup and straight down onto the top of the pastry, with no side profile of either visible. No metal tray anywhere -- both items stand directly on the black table. At the center of the table, referencing the pot sample photo, sits one small potted plant with a round candle (about one-third the size of the pot) beside it, the candle enclosed in a clear transparent case/holder for safety.",
      "camera": "strictly vertical overhead, exactly 90 degrees from horizontal (camera pointing straight down, shooting directly into the mouth of the glass from directly above it) -- positioned so that ONLY the top surface of the beverage (its rim, ice, and surface garnish, looking straight down into the drink) and the top surface of the food prop are visible. No side profile of the glass body, no side profile of the pastry, no wall, no glass wall -- a true straight-down camera angle for every element on the table.",
      "framing": "Follow the layout_sample photo's exact arrangement, a loose DIAMOND shape on the black table (not a straight line, not a simple triangle): (1) the potted plant sits upper-center-left; (2) the candle (in its clear transparent case) sits to the plant's right, at roughly the same height or very slightly lower; (3) the beverage (seen only from directly above -- its rim and surface) sits lower-left, below and slightly left of the plant, at a clearly lower position on the table than the plant/candle pair; (4) the food plate (croissant-type pastry or cake with fork and knife laid fully visible beside it, handles toward the viewer) sits lower-right, below and slightly right of the candle, roughly at the same height as the beverage but offset to the right -- larger in footprint than the cup, with nothing underneath the plate. Together the four elements form a diamond/quadrant layout: plant+candle occupy the upper pair, beverage+plate occupy the lower pair, with clear left-right offset AND up-down offset between every element -- never a single row or column. All four elements together occupy a MINORITY of the frame (roughly 35-45% of total frame area), with the plain black table filling a clearly larger background proportion around and between them.",
      "environment_override": {
        "wall_or_background": "no wall is visible at all in this strict vertical overhead angle -- only the plain black round table fills the frame, with perhaps the faintest soft-focus hint of a dark chair leg or floor right at the extreme edge of frame.",
        "surface": "a black round bistro table (or the visible portion of one), matte finish, filling most of the frame with generous bare table space around the beverage and plate."
      },
      "props": [
        "one croissant-type pastry or cake (top view only), plated on a small black ceramic plate per the prop_plating rule",
        "one simple fork and knife laid fully visible beside the pastry on the same plate, handles toward the viewer",
        "one small potted plant at the table center (top view), referencing the pot sample photo",
        "one round pillar candle (about 1/3 the size of the pot) beside the plant, fully enclosed in a clear transparent case/holder -- no exposed flame"
      ],
      "negative_extra": "No marble or white/light stone surface. No metal tray anywhere. No cookie or biscuit as the food prop -- it must be one croissant-type pastry or cake, top view only. No side profile of the cup, the glass body, or the pastry -- strictly top-down/straight-down for every element. No tight crop -- the black table background must occupy a clearly larger share of the frame than the combined props. No branded object, no cosmetic-looking prop, no reference-photo text or logo other than the user's own beverage cup logo. CRITICAL: the candle must NEVER show an open/exposed flame or bare wick -- it must always be shown fully enclosed in a clear transparent case/holder for safety. No bright, airy, or well-lit look on the table itself -- keep the tabletop dark and low-key. CRITICAL LAYOUT RULE: follow the diamond/quadrant arrangement above exactly -- plant+candle as the upper pair, beverage+plate as the lower pair, each of the four elements offset from every other element on both axes. Never arrange all four in a single straight line or column.",
      "uses_shared_layout": false
    },
    "handheld": {
      "display_name": "손으로 든 샷 (온도별 구도 분기)",
      "description": "Two distinct compositions depending on the beverage's temperature, both set in the same black-table cafe-lounge context referenced from black_table_chair.png, selected at prompt-build time via beverage_temp.",
      "hand_description": "a single, clearly feminine hand (slender fingers, smooth clean nails, no rings or jewelry) holds or rests near the cup. A shirt or sweater cuff/sleeve fully covers the wrist and lower forearm, so no bare wrist skin is visible at all -- the sleeve hem sits right at or past the wrist joint. No face, no other body part, and no identifying marks (tattoos, scars) are visible -- hand and sleeve only.",
      "environment_override_by_temp": {
        "hot": {
          "wall_or_background": "no tall wall plane -- the setting is a small black bistro table seen from the same seated-in-a-wingback-armchair vantage point as the aerial shot, with the same soft out-of-focus hint of other black furniture at the extreme frame edges.",
          "surface": "a small matte black bistro table with slim black metal legs.",
          "composition": "The hot beverage in its mug rests STILL on the black table, not lifted into the air. The hand described above rests gently on or beside the mug (e.g. wrapped partway around it) as if about to pick it up, but the mug's base stays in contact with the table."
        },
        "ice": {
          "wall_or_background": "the SAME tall, bare, rough-troweled dark-grey concrete wall used in the close_up shot -- a deliberate return to the concrete backdrop specifically for this iced handheld composition, not the black table interior used for the hot version or the aerial shot.",
          "surface": "only a sliver of the grey stone counter edge at the very bottom of frame, with a small prop (e.g. cookies) plated on a small black ceramic plate resting on it.",
          "composition": "The iced beverage in its glass is lifted OFF the table, held up in the air by the hand at roughly chest-to-shoulder height. CRITICAL: the glass must be held perfectly UPRIGHT and VERTICAL -- its base and rim stay level and straight, with no tilt, lean, or angle in any direction, as if it were still standing on an invisible flat surface. Do not tilt the glass toward the camera or to either side. The dark grey concrete wall fills the soft-focus background behind the raised glass."
        }
      },
      "camera": "50-70mm equivalent, near eye-level to slightly above, camera close to the subject (~30-50cm).",
      "props": {
        "hot": [],
        "ice": []
      },
      "negative_extra": "CRITICAL: this shot must include NO food prop of any kind -- no pastry, cake, cookie, plate, or napkin anywhere in frame. Only the beverage and the hand/sleeve holding or resting near it. No visible logo, wordmark, or embossed brand text anywhere. No metal tray anywhere. No identifiable person, face, or distinguishing personal detail -- hand and sleeve only, generic and unbranded. No bare wrist skin visible under any circumstance -- the cuff must cover it. No bright, airy, or well-lit look -- keep the whole frame dark and low-key.",
      "uses_shared_layout": false
    }
  },
  "cup_shapes": {
    "hot": {
      "display_name": "따뜻한 음료 - 손잡이 있는 다크그레이 머그컵 + 잔받침",
      "vessel": "a matte ceramic mug WITH a handle: a simple curved loop handle on one side, straight or gently tapered cylindrical body, thick smooth ceramic walls, medium height, sitting on a saucer beneath it -- a saucer/coaster is ALWAYS present under a hot beverage, in every single shot type (medium, close_up, aerial, and handheld alike), with NO exceptions. The mug and its saucer are always a MATCHING SET -- same exact color tone and same matte ceramic material as each other, never mismatched. SCOPE-LIMITED RULE: the mug+saucer ceramic color must be BLACK or DARK CHARCOAL-GREY ONLY, in every one of the four shot types without exception, regardless of what color the cup happens to be in the user's original uploaded product photo -- white is explicitly banned as a mug/saucer color anywhere in this preset. This white-ban applies ONLY to the mug and saucer ceramic material itself. It does NOT apply to anything else in the scene: the candle's wax may still be its natural white/ivory color, the potted plant and its pot may still be their natural light/grey/ivory tones, the beverage's own milk or cream color stays whatever it naturally is, and any naturally light-toned patch of the concrete wall or counter is unaffected -- only the mug and saucer are restricted to black/dark-grey.",
      "reference_note": "dark-grey handled mug + saucer, fixed by explicit user instruction (overrides the earlier plain white handleless 'hot_mug' reference)."
    },
    "ice": {
      "display_name": "아이스 음료 - 유리컵",
      "vessel": "a clear DOUBLE-WALL glass tumbler (two layers of glass with a visible air gap between them, exactly like the ice_glass reference photo) -- this double-wall structure is REQUIRED, not optional, and is the cup's defining, distinguishing feature versus any ordinary single-wall glass. The glass body must be THICK and SUBSTANTIAL, not thin or slender -- a sturdy, chunky double-wall tumbler with real visible wall thickness, not a delicate narrow glass. The rim and every double-wall edge line must be CRISP, SHARP, and clearly defined, with real glass reflections and highlights along the rim -- never soft, blurred, or low-contrast in a way that makes the glass look like it's covered by an invisible protective film; it must read unmistakably as real, clear glass. The beverage's color layers, condensation, and ice cubes are visible through the inner glass wall, with the outer wall's air gap creating a subtle but clearly visible separation line around the glass. Straight or gently tapered cylindrical shape, no handle.",
      "reference_note": "shape derived from the 'ice_glass' reference photo: a clear double-wall glass tumbler where the double-wall air gap is clearly visible, showing swirled color layers and ice cubes through the inner glass.",
      "no_straw": true,
      "no_saucer": true
    }
  },
  "prop_plating": {
    "rule": "There is NO metal tray in any shot, and NO napkin of any kind in any shot -- napkins are removed from this preset entirely. The food prop -- always one croissant-type pastry or one small cake, never a cookie or biscuit -- is plated together with a simple fork and knife directly on one small plate, with nothing underneath the plate. The plate must be PORCELAIN or CERAMIC ONLY -- plastic, melamine, or any synthetic plate material is STRICTLY FORBIDDEN. The plate's color must be BLACK or DARK CHARCOAL-GREY (matte, unbranded, simple round or gently rectangular shape). When the beverage is HOT, the plate's color must be the OPPOSITE choice from whichever color the mug+saucer set ends up being in that shot -- if the mug+saucer are black, the plate must be dark charcoal-grey, and if the mug+saucer are dark charcoal-grey, the plate must be black -- so the plate never matches the mug+saucer set. When the beverage is ICED, either black or dark-grey is fine for the plate with no such matching constraint. FORK AND KNIFE PLACEMENT (critical): the fork and knife must lie fully visible on the plate, beside the pastry -- NEVER underneath the pastry, NEVER partially hidden or covered by it. Their handles must generally point toward the camera/viewer (the near edge of the frame), with the tines and blade end pointing away from the viewer, toward the far side of the plate. NATURAL PLACEMENT: the fork and knife must NOT lie perfectly parallel to each other like two rigid straight lines (avoid a stiff, robotic '11'-shaped alignment) -- give them a slight natural angle or crossing offset relative to each other, as if casually set down by hand, while still keeping both handles generally toward the viewer. EXTENT ON THE PLATE: at least the head (tines/blade) AND roughly the first half of the handle must rest on top of the plate -- not just the head. The remaining (far) half of the handle may extend past the plate's rim toward the viewer, but NO MORE than half of the handle's total length may hang off the plate -- the fork and knife must never dangle mostly off the edge, and never rest with only their very tip touching the plate."
  },
  "beverage_rules": {
    "no_straw_for_ice": "CRITICAL, applies to every shot type: an iced beverage must NEVER include a straw of any kind -- no straw, no lid with a straw hole, no straw wrapper. The drink is sipped directly from the glass rim.",
    "no_saucer_for_ice": "CRITICAL, applies to every shot type: an iced beverage must NEVER sit on a saucer, coaster, or plate underneath the glass -- the glass sits directly on the table/counter/windowsill surface with nothing beneath it (no tray, no coaster, no saucer)."
  },
  "container_modes": {
    "reference_cup": {
      "display_name": "레퍼런스 컵 버전",
      "description": "The cup/vessel's shape, material, color, handle, and lid design ALWAYS follow this preset's own cup_shapes definition (selected by beverage_temp), never the user's original cup. Only the beverage's own visual content -- its color, ice, foam, layering -- comes from the user's photo. ONE input photo is used for all four shot types (medium, close_up, aerial, handheld) -- ideally cropped or framed so the cup's own body/logo is not too dominant in the frame, so the model has minimal cup-shape signal to anchor on. The output camera angle for each shot (including aerial's strict vertical, top-surface-only framing) is fully controlled by that shot's own text prompt and does not depend on the input photo's angle.",
      "logo_policy": "no logo is expected or added -- the cropped input shows no cup, so the reference cup renders with no logo unless the target brand contract separately specifies one."
    },
    "user_cup": {
      "display_name": "사용자 컵 버전",
      "description": "The ENTIRE original uploaded photo (cup and beverage together) is sent to the model. The cup's shape, material, color, handle/lid design, and any logo are reconstructed faithfully from the user's own photo -- this preset's cup_shapes definitions (mug/glass rules, black/dark-grey color rule, saucer rule, etc.) do NOT apply in this mode, since the user's own cup design is the source of truth.",
      "logo_policy": "preserve any real logo/wordmark exactly as it appears in the user's uploaded photo; never invent a new one."
    },
    "shared_rule": "Regardless of which container_mode is selected, the beverage's own type and temperature (hot vs iced, and its recipe/color/ice-or-foam state) ALWAYS come from the user's own uploaded content, never from any reference photo or preset assumption. A reference cup's own original beverage or hot/iced state (if the reference happened to depict one) is never followed."
  }
}
''')

print(PRESET["display_name"])
print("preset_id:", PRESET["preset_id"])
print("shot_variants:", list(PRESET["shot_variants"].keys()))


## 4. 프롬프트 빌더 (구도별 + 온도별 + 컵 모드별)

각 구도(`shot_variants`)가 자기만의 배경(`environment_override`)을 가집니다 (미디엄=글래스월 창가, 항공샷/손샷=블랙 테이블+소파석 시선, close_up=기존 그레이 콘크리트).
`beverage_temp`("hot"/"ice")는 이제 **사용자가 직접 고르지 않고, 업로드한 사진에서 자동 판별**합니다 (섹션 5-1 참고).
새로 추가된 **`container_mode`**가 컵 디자인의 출처를 결정합니다:
- `"reference_cup"` — 컵 모양/재질/색/손잡이/뚜껑은 이 프리셋의 레퍼런스 컵(hot=다크 머그, ice=유리컵) 디자인을 따르고, **음료 내용물(색/얼음/거품/온도 상태)만 사용자 사진에서** 가져옵니다.
- `"user_cup"` — 사용자가 업로드한 사진의 컵과 음료를 통째로 참고해서, **컵 디자인도 원본 그대로 재구성**합니다 (이 경우 프리셋의 머그 색상/손잡이/받침 규칙은 적용되지 않습니다).
어느 모드든 **음료 자체의 종류·온도 상태는 항상 사용자 사진이 우선**이고, 레퍼런스 사진에 담긴 음료나 온도는 절대 따르지 않습니다.

In [ ]:
def build_prompt(
    preset: dict,
    shot_variant: str,
    beverage_temp: str = "ice",
    container_mode: str = "user_cup",
    product_note: str = "",
    extra_notes: str = "",
) -> str:
    """preset(dict) + shot_variant(str) + beverage_temp("hot"|"ice") + container_mode로 상세한 텍스트 프롬프트를 조립한다.

    shot_variant: "medium" | "close_up" | "aerial" | "handheld"
    beverage_temp: "hot" | "ice" -- handheld 구도 분기 및 (reference_cup 모드일 때) 컵 모양 선택에 사용
    container_mode: "reference_cup" | "user_cup" -- 컵 디자인을 프리셋 레퍼런스에서 가져올지, 사용자 원본에서 가져올지
    """
    if shot_variant not in preset["shot_variants"]:
        raise ValueError(f"Unknown shot_variant: {shot_variant!r}. Choose from {list(preset['shot_variants'])}")
    if beverage_temp not in ("hot", "ice"):
        raise ValueError(f"beverage_temp must be 'hot' or 'ice', got {beverage_temp!r}")
    if container_mode not in ("reference_cup", "user_cup"):
        raise ValueError(f"container_mode must be 'reference_cup' or 'user_cup', got {container_mode!r}")

    variant = preset["shot_variants"][shot_variant]
    light = preset["lighting"]
    color = preset["color"]
    scene = preset["scene"]
    capture = preset["capture"]
    tone = preset["tone_contract"]
    blocks = preset["prompt_blocks"]
    lock = preset["preservation_policy"]
    gates = preset["quality_gates"]
    cup = preset["cup_shapes"][beverage_temp]
    plating_rule = preset["prop_plating"]["rule"]
    modes = preset["container_modes"]

    # handheld는 온도별로 environment_override / props가 dict로 갈라져 있음
    env_override = variant.get("environment_override")
    if env_override is None and "environment_override_by_temp" in variant:
        env_override = variant["environment_override_by_temp"][beverage_temp]

    props = variant.get("props")
    if isinstance(props, dict):
        props = props[beverage_temp]

    lines = []
    lines.append("### ROLE")
    lines.append(
        "You are compositing ONE real user beverage photo into a brand-new, fully "
        "photorealistic cafe environment. The output must look like a single unedited "
        "phone/camera photo taken in one real room under one real light source -- "
        "never a collage, never a sticker-on-background look."
    )

    lines.append("\n### INPUT IMAGE MODE: " + modes[container_mode]["display_name"] + f" ({container_mode})")
    lines.append(modes[container_mode]["description"])
    lines.append(modes["shared_rule"])

    lines.append("\n### SHOT TYPE: " + variant["display_name"] + f" ({shot_variant}, {beverage_temp})")
    lines.append(variant["description"])
    if "hand_description" in variant:
        lines.append("Hand: " + variant["hand_description"])
    if env_override and "composition" in env_override:
        lines.append(env_override["composition"])

    lines.append("\n### OVERALL LOOK (shared mood across shot types)")
    lines.append(blocks["look"])
    lines.append(capture["look"] + ".")
    lines.append("Realism requirement: " + capture["realism"] + ".")

    lines.append("\n### ENVIRONMENT: WALL/BACKGROUND + SURFACE")
    if env_override:
        lines.append(env_override["wall_or_background"] + ".")
        lines.append("Surface: " + env_override["surface"] + ".")
        if "lighting_note" in env_override:
            lines.append(env_override["lighting_note"] + ".")
    else:
        lines.append(scene["background"] + ".")
        lines.append(
            "Below the wall, on a soft horizontal seam, there is " + scene["surface"] + ". "
            "This surface is a different, slightly coarser grey texture family than the wall above it."
        )

    lines.append("\n### CAMERA + FRAMING FOR THIS SHOT")
    lines.append("Camera: " + variant["camera"])
    if "framing" in variant:
        lines.append(variant["framing"])

    lines.append("\n### CUP / VESSEL SHAPE")
    if container_mode == "reference_cup":
        lines.append(cup["vessel"])
        if beverage_temp == "hot" and "color_by_shot" in cup:
            color_rule = cup["color_by_shot"].get(shot_variant, cup["color_by_shot"]["other"])
            lines.append(color_rule)
        lines.append(
            "This is the PRESET's own reference cup design -- it is deliberately NOT the user's original cup. "
            "The input image has been cropped down to the beverage's own content (its color, ice, foam, "
            "layering), with little to no original cup structure visible, specifically so this reference cup "
            "design can be applied cleanly without fighting the original cup's shape."
        )
    else:  # user_cup
        lines.append(
            "Reconstruct the EXACT cup/container shown in the user's uploaded photo -- its shape, material, "
            "color, handle (if any), and lid (if any) -- faithfully and natively for this new camera and "
            "light. Do NOT apply this preset's reference mug/glass color, handle, or saucer rules in this "
            "mode -- the user's own original cup design is the source of truth for the vessel's appearance."
        )
    lines.append(
        "Preserve any real logo or wordmark exactly as it appears on the user's original cup photo, rendered "
        "clearly and legibly on the reconstructed vessel. NEVER invent, generate, or design a new logo of any "
        "kind. If no logo is visible in the input image, the output must also show no logo -- do not add one."
    )
    if cup.get("no_straw"):
        lines.append(preset["beverage_rules"]["no_straw_for_ice"])
    if cup.get("no_saucer") and container_mode == "reference_cup":
        lines.append(preset["beverage_rules"]["no_saucer_for_ice"])

    # 이 구도가 자기만의 조명(lighting_note)을 갖고 있으면(예: medium의 저녁 실내조명),
    # 공용 '자연광' 섹션은 건너뛴다 -- 안 그러면 '자연광이다'와 '실내조명이다'가 동시에 들어가 모순됨
    has_own_lighting = bool(env_override and "lighting_note" in env_override)
    if not has_own_lighting:
        lines.append("\n### LIGHTING (shared natural light unless this shot overrides it above)")
        lines.append(
            light["type"] + ", arriving from the " + light["direction"]
            + " (azimuth ~" + str(light["azimuth_degrees"]) + " degrees, elevation ~"
            + str(light["elevation_degrees"]) + " degrees). Intensity: " + light["intensity"]
            + ". Contrast: " + light["contrast"] + ". Shadow behaviour: " + light["shadow"] + ". "
            + light["highlight_control"] + "."
        )

    lines.append("\n### COLOR & GRADE (shared)")
    lines.append(
        "White balance approximately " + str(color["white_balance_kelvin"]) + "K, " + color["temperature"] + ". "
        "Black point: " + color["black_point"] + ". Contrast character: " + color["contrast"] + ". "
        "Saturation: " + color["saturation"] + ". "
        "Approximate palette (for reference, do not render as flat color blocks): "
        + ", ".join(color["palette_hex"]) + "."
    )

    lines.append("\n### TEXTURE / MATERIALS (shared)")
    lines.append(scene["texture"] + ".")
    lines.append("Fine-grain realism: " + capture["grain"] + ".")
    lines.append("Include tiny natural imperfections such as: " + "; ".join(capture["imperfections"]) + ".")

    lines.append("\n### PROPS FOR THIS SHOT")
    if props:
        lines.append("Use only: " + "; ".join(props) + ".")
        lines.append("Prop plating rule (always applies): " + plating_rule)
    else:
        lines.append(
            "CRITICAL: this shot must include NO food prop of any kind -- no bread, pastry, cake, cookie, "
            "plate, or napkin anywhere in frame. Only the beverage (and, if this is a handheld shot, the hand "
            "holding or resting near it) appear."
        )

    lines.append("\n### PRODUCT / IDENTITY PRESERVATION (hard constraints, shared)")
    lines.append("- " + "\n- ".join(lock["hard_lock"]))
    lines.append(blocks["preservation"])

    lines.append("\n### WHAT TO AVOID (shared)")
    lines.append(blocks["negative"])
    lines.append("Forbidden surface readings: " + ", ".join(tone["forbidden_surface_reading"]) + ".")
    if "negative_extra" in variant:
        lines.append("Shot-specific: " + variant["negative_extra"])

    lines.append("\n### QUALITY BAR (self-check before finalizing)")
    lines.append("- " + "\n- ".join(gates.values()))

    if product_note:
        lines.append("\n### PRODUCT NOTE FROM USER")
        lines.append(product_note)

    if extra_notes:
        lines.append("\n### REVISION NOTES (apply these on top of everything above)")
        lines.append(extra_notes)

    return "\n".join(lines)


# 미리보기 -- 이 미리보기 전용 변수들은 실제 생성에 쓰이는 전역 BEVERAGE_TEMP/CONTAINER_MODE와 별개입니다.
# 실제 값들은 바로 다음 섹션들(4-1, 5-1)에서 설정합니다.
_preview_temp = "ice"
_preview_mode = "user_cup"
for variant_name in PRESET["shot_variants"]:
    preview = build_prompt(PRESET, variant_name, beverage_temp=_preview_temp, container_mode=_preview_mode, product_note="사용자가 업로드한 음료 사진 그대로 사용")
    print(f"=== {variant_name} / {_preview_temp} / {_preview_mode} ({len(preview)}자) ===")
    print(preview[:400], "...\n")


## 4-1. 컵 모드 설정 (`CONTAINER_MODE`) -- ⚠️ 컵 디자인의 출처를 정합니다

- `"reference_cup"` — 컵은 이 프리셋의 레퍼런스 디자인(hot=다크 손잡이 머그, ice=유리컵)을 따르고, 음료 내용물(색/얼음/거품)만 사용자 사진에서 가져옵니다.
- `"user_cup"` — 사용자의 원본 컵 디자인을 그대로 재구성합니다 (기존 방식과 동일).

**두 모드 다 사진은 1장만** 올리면 됩니다 (구도별 카메라 각도는 사진이 아니라 텍스트 프롬프트가 결정하므로, 사진을 여러 장 준비할 필요가 없습니다).
`BEVERAGE_TEMP`와 마찬가지로 이 값도 재실행에 안전하도록 독립 셀에 있습니다.

In [ ]:
CONTAINER_MODE = "user_cup"  # "reference_cup" 또는 "user_cup"

print(f"CONTAINER_MODE = {CONTAINER_MODE!r} 로 설정되었습니다. (사진은 다음 셀에서 1장만 올리면 됩니다)")


## 5. 제품 이미지 업로드

이제 `CONTAINER_MODE`와 무관하게 **사진 1장**만 올리면 됩니다 (4가지 구도 전부 이 한 장을 씁니다).

`reference_cup` 모드일 때는, 컵 몸통/로고가 너무 크게 안 보이도록 살짝 크롭해서 올리면 프리셋 컵으로 더 안정적으로 바뀝니다 (필수는 아니고 권장 사항입니다). 각 구도의 카메라 각도(항공샷의 수직 탑다운 포함)는 사진이 아니라 텍스트 프롬프트가 결정하므로, 사진의 촬영 각도는 크게 상관없습니다.

In [ ]:
from google.colab import files
import io
from PIL import Image

print("사진 1장을 올려주세요.")
uploaded = files.upload()
PRODUCT_IMAGE_PATH = next(iter(uploaded.keys()))
print("업로드된 제품 이미지:", PRODUCT_IMAGE_PATH)
display(Image.open(PRODUCT_IMAGE_PATH))


## 5-1. 음료 온도 자동 판별 (`BEVERAGE_TEMP`) -- 완전 자동, 확인창 없음

방금 올린 사진을 Gemini로 한 번 분석해서 조용히 `"hot"` 또는 `"ice"`를 정합니다 (Gemini 1회 호출). **사용자에게 확인을 묻지 않고 그대로 다음 단계로 진행됩니다.**
혹시 온도가 잘못 반영된 결과가 나오면, 그건 사전 확인창이 아니라 **8번 Vision QA 단계에서 사후에 자동으로 잡아냅니다** (`fail`/`needs_review`로 표시됨). 그래도 직접 강제로 바꾸고 싶다면 이 셀 실행 직후 `BEVERAGE_TEMP = "hot"`처럼 다시 대입하고 아래 셀부터 다시 실행하면 됩니다.

In [ ]:
from PIL import Image as PILImage

QA_MODEL = "gemini-3.1-flash-lite"  # 자동 온도 판별 + (섹션 8) Vision QA에 공통으로 사용

def detect_beverage_temp(image_path) -> str:
    """업로드된 사진 하나를 보고 이 음료가 hot인지 ice인지 Gemini로 1회 판별한다."""
    img = PILImage.open(image_path)
    detect_prompt = (
        "Look at this beverage photo. Decide whether the drink is HOT or ICED. "
        "Look for cues like: visible ice cubes, condensation on the glass, a tall clear glass (usually iced) "
        "vs. steam, a mug/cup with no ice, foam/latte-art on top with no ice (usually hot). "
        "Respond with ONLY one lowercase word: \"hot\" or \"ice\". No other text."
    )
    response = gemini_client.models.generate_content(
        model=QA_MODEL,
        contents=[img, detect_prompt],
        config=genai_types.GenerateContentConfig(temperature=0),
    )
    guess = response.text.strip().lower()
    return "hot" if "hot" in guess else "ice"


BEVERAGE_TEMP = detect_beverage_temp(PRODUCT_IMAGE_PATH)
print(f"BEVERAGE_TEMP = {BEVERAGE_TEMP!r} (자동 판별, 조용히 다음 단계로 진행)")


## 6. 이미지 생성 (OpenAI `gpt-image-2`, medium 품질, 1회 호출)

힉스필드 경유 없이 OpenAI에 직접 요청합니다. 품질은 `medium`으로 고정합니다 (원하면 `IMAGE_QUALITY`를 `"low"`/`"high"`로 바꿔서 품질·비용을 비교해볼 수 있어요).
크기는 세로(4:5에 가장 가까운) `1024x1536`을 기본값으로 둡니다.

In [ ]:
import base64
import time
from pathlib import Path

IMAGE_MODEL = "gpt-image-2"
IMAGE_QUALITY = "medium"           # "low" / "medium" / "high" 중 선택 (medium 기준)
IMAGE_SIZE = "1024x1536"           # 허용값: 1024x1024 / 1024x1536 / 1536x1024

OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def generate_image(prompt: str, product_image_path: str, run_tag: str = "run") -> Path:
    """제품 이미지 + 프롬프트로 이미지를 1회 생성하고 로컬에 저장한다. (OpenAI 1회 호출, 재시도 루프 없음)"""
    with open(product_image_path, "rb") as image_file:
        result = openai_client.images.edit(
            model=IMAGE_MODEL,
            image=image_file,
            prompt=prompt,
            size=IMAGE_SIZE,
            quality=IMAGE_QUALITY,
        )

    b64_data = result.data[0].b64_json
    image_bytes = base64.b64decode(b64_data)

    timestamp = time.strftime("%Y%m%d-%H%M%S")
    out_path = OUTPUT_DIR / f"{run_tag}_{timestamp}.png"
    out_path.write_bytes(image_bytes)
    return out_path


## 7. 첫 생성 실행 (4가지 구도, 딱 4번만 생성)

`PRESET["shot_variants"]`에 있는 **정확히 4개** 구도(medium / close_up / aerial / handheld)에 대해서만 1장씩 생성합니다.
개수가 고정된 `for` 루프라서 몇 번 호출될지 항상 예측 가능합니다 (OpenAI 4회 호출, 무한/재시도 루프 아님).
`BEVERAGE_TEMP`(자동 판별됨)와 `CONTAINER_MODE`(위에서 지정)를 그대로 씁니다. 이제 입력 이미지는 `CONTAINER_MODE`/구도와 무관하게 항상 업로드한 사진 1장(`PRODUCT_IMAGE_PATH`)입니다.

In [ ]:
SHOT_VARIANTS_TO_RUN = ["medium", "close_up", "aerial", "handheld"]  # 정확히 이 4개만, 늘리려면 여기서 직접 수정

def get_input_image_for_shot(shot_variant: str) -> str:
    """입력 이미지 경로를 고른다. 지금은 4구도·양쪽 CONTAINER_MODE 모두 동일한 사진 1장을 쓴다."""
    return PRODUCT_IMAGE_PATH


generated_images = {}  # shot_variant -> Path

for shot_variant in SHOT_VARIANTS_TO_RUN:  # 고정 4회 반복 (무한/재시도 루프 아님)
    prompt = build_prompt(
        PRESET,
        shot_variant,
        beverage_temp=BEVERAGE_TEMP,
        container_mode=CONTAINER_MODE,
        product_note="사용자가 업로드한 음료 사진의 색/온도/얼음 상태를 그대로 유지",
    )
    input_image = get_input_image_for_shot(shot_variant)
    out_path = generate_image(prompt, input_image, run_tag=f"dark_grey_metal_{shot_variant}_{BEVERAGE_TEMP}")
    generated_images[shot_variant] = out_path
    print(f"[{shot_variant}/{BEVERAGE_TEMP}/{CONTAINER_MODE}] 생성 완료:", out_path)
    display(Image.open(out_path))


## 8. Vision QA (Gemini `gemini-3.1-flash-lite`로 자동 점검)

생성된 이미지를 프리셋의 `quality_gates` / `hard_lock` / `failure_recovery` 기준으로 자동 점검합니다.
사람이 최종 판단하기 전, 명백한 실패(트레이 없음, 벽이 밝고 광택남, 텍스트 노출 등)를 빠르게 걸러내는 용도입니다.
팀 파이프라인의 `configs/evaluator.json`과 동일하게 **`gemini-3.1-flash-lite`**를 사용하고, `response_mime_type="application/json"`으로 구조화된 JSON을 강제로 받습니다.
이 호출은 이미지 생성과 별개의 API(Gemini)라서, OpenAI 사용량에는 전혀 영향을 주지 않습니다.

In [ ]:
from PIL import Image as PILImage

QA_MODEL = "gemini-3.1-flash-lite"

def vision_qa(image_path: Path, preset: dict, beverage_temp: str = None) -> dict:
    """생성된 이미지를 프리셋 기준으로 Gemini에게 점검시키고 JSON으로 결과를 받는다. (Gemini 1회 호출, 재시도 루프 없음)

    beverage_temp을 넘기면, 온도 관련 확인은 여기(사후 QA)에서만 이뤄진다 -- 생성 전에 사용자에게
    따로 확인을 묻지 않고, 자동판별 결과가 틀렸을 경우를 이 체크리스트가 사후에 잡아낸다.
    """
    generated_image = PILImage.open(image_path)

    checklist = []
    checklist.extend(preset["quality_gates"].values())
    checklist.extend(preset["preservation_policy"]["hard_lock"])
    checklist.append("Negative space (bare wall) above the product should be roughly "
                      f'{int(preset["subject_layout"]["negative_space"]["top_ratio"]*100)}% of frame height or more.')

    if beverage_temp == "ice":
        checklist.append(
            "This should be an ICED beverage: check for ice cubes and/or condensation, and NO hot-drink cues "
            "(steam, latte-art foam with no ice, a mug with no ice). If it looks like a hot beverage instead "
            "of iced, mark this item as fail."
        )
    elif beverage_temp == "hot":
        checklist.append(
            "This should be a HOT beverage: check for a mug with no ice cubes and no cold-glass condensation. "
            "If ice cubes or a clear glass with ice appear instead, mark this item as fail."
        )
    else:
        checklist.append(
            "Check that the beverage's temperature cues (ice cubes/condensation vs. steam/latte-art foam) "
            "are internally consistent and not contradictory -- never ice cubes together with hot-drink foam art."
        )

    qa_prompt = (
        "You are a strict photo QA reviewer for an Instagram cafe-ad pipeline. "
        "Check the attached image against EVERY item below. "
        "Respond ONLY with valid JSON in this exact shape:\n"
        '{"overall": "pass" | "needs_review" | "fail", '
        '"checks": [{"item": str, "status": "pass"|"fail", "note": str}], '
        '"revision_notes": str}\n\n'
        "Checklist:\n- " + "\n- ".join(checklist)
    )

    response = gemini_client.models.generate_content(
        model=QA_MODEL,
        contents=[generated_image, qa_prompt],
        config=genai_types.GenerateContentConfig(
            response_mime_type="application/json",
            temperature=0,
        ),
    )

    try:
        return json.loads(response.text)
    except json.JSONDecodeError:
        return {"overall": "unparsed", "checks": [], "revision_notes": response.text}


qa_result = vision_qa(generated_images["medium"], PRESET, beverage_temp=BEVERAGE_TEMP)  # 데모: medium 구도 1장만 먼저 확인
print(json.dumps(qa_result, ensure_ascii=False, indent=2))


### 8-1. 4장 전체 QA (딱 4번만 호출)

방금 생성한 4개 구도 각각에 대해 QA를 실행합니다. 역시 정해진 4번만 호출됩니다 (Gemini 4회 호출, 무한 루프 아님).

In [ ]:
qa_results = {}  # shot_variant -> qa dict

for shot_variant, out_path in generated_images.items():  # 고정 4회 반복
    qa_results[shot_variant] = vision_qa(out_path, PRESET, beverage_temp=BEVERAGE_TEMP)
    print(f"=== {shot_variant} QA ===")
    print(json.dumps(qa_results[shot_variant], ensure_ascii=False, indent=2))


## 9. 반복 개선 루프

QA 결과나 직접 눈으로 확인한 피드백을 `revision_notes`에 넣고, 어떤 구도(`shot_variant`)와 온도(`beverage_temp`)를 다시 만들지 골라서 재생성합니다.
자연스러운 결과가 나올 때까지 이 셀만 반복 실행하면 됩니다 (호출할 때마다 정확히 생성 1번 + QA 1번, 총 2번의 API 콜). 모든 시도는 `run_manifest.jsonl`에 기록됩니다.

In [ ]:
MANIFEST_PATH = OUTPUT_DIR / "run_manifest.jsonl"

def iterate(shot_variant: str, revision_notes: str, beverage_temp: str = None, container_mode: str = None) -> Path:
    """revision_notes를 해당 구도 프롬프트에 덧붙여 재생성하고, 결과를 QA한 뒤 manifest에 기록한다."""
    beverage_temp = beverage_temp or BEVERAGE_TEMP
    container_mode = container_mode or CONTAINER_MODE
    prompt = build_prompt(
        PRESET,
        shot_variant,
        beverage_temp=beverage_temp,
        container_mode=container_mode,
        product_note="음료의 레시피/색/온도/얼음 상태는 절대 바꾸지 말 것",
        extra_notes=revision_notes,
    )
    input_image = get_input_image_for_shot(shot_variant)
    out_path = generate_image(prompt, input_image, run_tag=f"dark_grey_metal_{shot_variant}_{beverage_temp}")
    qa = vision_qa(out_path, PRESET, beverage_temp=beverage_temp)
    generated_images[shot_variant] = out_path  # 최신 결과로 갱신

    with open(MANIFEST_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "shot_variant": shot_variant,
            "beverage_temp": beverage_temp,
            "container_mode": container_mode,
            "output": str(out_path),
            "revision_notes": revision_notes,
            "qa": qa,
        }, ensure_ascii=False) + "\n")

    print(f"[{shot_variant}/{beverage_temp}/{container_mode}] 생성 완료:", out_path)
    display(Image.open(out_path))
    print(json.dumps(qa, ensure_ascii=False, indent=2))
    return out_path

# 사용 예시 (필요할 때마다 shot_variant / revision_notes만 바꿔서 다시 실행하세요):
# iterate("aerial", "빵이 트레이 밖으로 살짝 나가 있어요. 트레이 안쪽으로 옮기고 살짝 더 크게 보이게 해주세요.")


## 10. 결과 다운로드

지금까지 생성된 모든 이미지와 `run_manifest.jsonl`을 zip으로 묶어 다운로드합니다.

In [ ]:
import shutil
from google.colab import files as gfiles

zip_path = shutil.make_archive("/content/dark_grey_metal_outputs", "zip", OUTPUT_DIR)
gfiles.download(zip_path)


## 지금까지의 작업 이력 요약 + 남은 확인 사항

이 프리셋은 실제 생성 결과를 보면서 여러 차례 반복 수정됐습니다. 주요 변경 이력:
- 트레이 완전 제거 → 음료는 표면에 직접, 빵/케익은 도자기 접시+포크나이프로
- 냅킨 완전 제거, 포크·나이프 자연스러운 배치 규칙 추가
- `medium`을 그레이 콘크리트 → 글래스월 창가 → 1층 바 좌석(자연광)으로 재설계
- `aerial`을 세션 중 소파-POV → 완전 수직 탑다운으로 되돌리고, 화분+초+빵접시 다이아몬드 배치
- `handheld`를 온도별로 분기(hot=테이블 위, ice=공중에 들고), 소품 전부 제거
- 아이스 컵에 더블월 유리 필수화(레퍼런스 컵과 사용자 컵이 구분 안 되던 문제 해결), 컵-테이블 접촉부 광 누출 금지
- `BEVERAGE_TEMP` 수동 선택 → Gemini 자동 판별로 전환, `CONTAINER_MODE`(reference_cup/user_cup) 신규 도입

**아직 확인이 필요한 부분**:
1. **`cup_shapes.hot`의 색 규칙**이 `close_up`만 화이트 허용하던 걸 전 구도 블랙/다크그레이로 통일했는데, 실제 생성에서 화이트가 완전히 사라졌는지는 이미지 편집 모델의 원본 anchoring 특성상 계속 지켜봐야 합니다.
2. **`reference_cup` 모드의 로고 정책** — 지금은 크롭된 입력에 로고가 안 보이면 "로고 없음"으로 자연스럽게 처리되는데, 나중에 특정 브랜드 로고를 레퍼런스 컵에 새로 얹고 싶은 요구가 생기면 `target brand contract`로 별도 분리하는 걸 추천드립니다.
3. **크롭 자동화** — `reference_cup` 모드에서 "컵이 덜 보이게 크롭"을 사람이 직접 해야 하는데, 나중에 여유 되면 Gemini로 자동 크롭 좌표를 뽑는 헬퍼를 추가할 수 있습니다.

이 중에 방향을 정해주시면 프리셋을 바로 반영할게요.
